# Chronological Train/Test Split

## Objective

This notebook creates a leakage-safe train/test split for
Top-N recommendation evaluation.

We will:
- Perform per-user chronological splitting
- Ensure no future information leaks into training
- Validate split integrity
- Export train/test datasets

All model statistics must be computed on TRAIN only.

## Imports and Configuration
Define split parameters.

In [4]:
import pandas as pd

TEST_RATIO = 0.2
MIN_TEST_INTERACTIONS = 1  # at least 1 interaction in test
RANDOM_STATE = 42

In [2]:
class Dataset():
    def __init__(self, path):
        self.path = path
        self.dataset = pd.read_csv(self.path)

## Load Processed Dataset

In [5]:
rating_dataset = Dataset("../data/processed/ratings_clean.csv")

In [6]:
rating_dataset.dataset.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,2000-07-30 18:45:03
1,1,3,4.0,2000-07-30 18:20:47
2,1,6,4.0,2000-07-30 18:37:04
3,1,47,5.0,2000-07-30 19:03:35
4,1,50,5.0,2000-07-30 18:48:51


## Splitting Strategy

We perform a per-user chronological split:

For each user:
- Sort interactions by time
- First (1 - TEST_RATIO) → Train
- Last TEST_RATIO → Test

This simulates real-world recommendation:
We train on past behavior and predict future behavior.

In [8]:
train_list = []
test_list = []

for user, user_df in rating_dataset.dataset.groupby("userId"):
    user_df = user_df.sort_values("timestamp")
    
    n_interactions = len(user_df)
    n_test = max(int(n_interactions * TEST_RATIO), MIN_TEST_INTERACTIONS)
    
    test_part = user_df.tail(n_test)
    train_part = user_df.iloc[:-n_test]
    
    # Only include users with at least 1 train interaction
    if len(train_part) > 0:
        train_list.append(train_part)
        test_list.append(test_part)

train_df = pd.concat(train_list)
test_df = pd.concat(test_list)

In [11]:
train_df.head()

,userId,movieId,rating,timestamp
73,1,1210,5.0,2000-07-30 18:08:19
43,1,804,4.0,2000-07-30 18:08:19
120,1,2018,5.0,2000-07-30 18:08:43
171,1,2628,4.0,2000-07-30 18:08:43
183,1,2826,4.0,2000-07-30 18:08:43


In [12]:
test_df.head()

,userId,movieId,rating,timestamp
176,1,2654,5.0,2000-07-30 18:56:33
174,1,2644,4.0,2000-07-30 18:56:33
76,1,1219,2.0,2000-07-30 18:56:33
91,1,1348,4.0,2000-07-30 18:56:33
180,1,2716,5.0,2000-07-30 18:56:54
